# v2 · Notebook 01 — Preparação dos dados (mMARCO-pt, split **dev**)

Corrige as duas fraquezas principais da v1 (ver `phase2/WEAKNESSES.md`):

1. **Vazamento**: a v1 avaliava com perguntas de **treino**, e o cross-encoder pronto foi treinado no mMARCO.
   Aqui a avaliação usa o conjunto oficial **dev small** (`portuguese_queries.dev.small.tsv` + `qrels.dev.small.tsv`).
2. **Corpus fácil**: a v1 tinha só ~12 negativos difíceis entre 5.312 passagens. Aqui as candidatas de cada pergunta
   são o **top-100 do BM25 oficial** (`run.bm25_portuguese-msmarco.txt`) sobre a **coleção completa** (8,8 milhões de passagens),
   ou seja, negativos difíceis por construção.

Também prepara os **pares de treino** (triplas consulta / positiva / negativa) para o notebook 02.

Roda **uma vez** (tempo estimado: 20–40 min, dominado pelo download de 3,4 GB da coleção; não precisa de GPU).
Tudo é salvo no Google Drive.

In [ ]:
!pip install -q ftfy==6.3.1

In [ ]:
# ==============================================================================
# Google Drive: tudo o que precisa sobreviver a uma desconexão fica aqui
# ==============================================================================
import os
from google.colab import drive
drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/AssignmentPLN_v2"
DATA = f"{BASE}/data"          # saída do notebook 01
CKPT = f"{BASE}/checkpoints"   # saída do notebook 02
RES  = f"{BASE}/results"       # saída do notebook 03
for d in (DATA, CKPT, RES):
    os.makedirs(d, exist_ok=True)

SEED = 42

In [ ]:
import json, random, time, platform
from collections import defaultdict
import numpy as np
import pandas as pd
import ftfy
from huggingface_hub import hf_hub_download

# ---------------- Configuração ----------------
N_DEV_QUERIES   = None     # None = TODAS as perguntas do dev small (6.980), como nos artigos; um número = amostra aleatória
DEPTH           = 100      # candidatas do BM25 guardadas por pergunta (= maior K reranqueado)
N_TRAIN_TRIPLES = 300_000  # triplas de treino amostradas (o notebook 02 pode usar menos)
N_VAL_TRIPLES   = 2_000    # triplas separadas para validação do treino (NÃO são o conjunto de teste)
FIX_TEXT        = True     # corrige mojibake (ex.: "PlazaÃ®") com ftfy nas consultas e passagens
# As triplas de treino só são preparadas se ainda não existirem no Drive. Elas não dependem do conjunto dev
# (semente própria), então mudar N_DEV_QUERIES NÃO exige rodar o notebook 02 de novo.
PREP_TRAIN      = not os.path.exists(f"{DATA}/train_triples.parquet")
print("preparar triplas de treino:", PREP_TRAIN)

REPO = "unicamp-dl/mmarco"
FILES = {   # caminhos conferidos no repositório em 2026-11-24
    "collection":    "data/google/collections/portuguese_collection.tsv",   # 3,4 GB
    "train_queries": "data/google/queries/train/portuguese_queries.train.tsv",
    "dev_queries":   "data/google/queries/dev/portuguese_queries.dev.small.tsv",
    "dev_qrels":     "data/qrels.dev.small.tsv",
    "bm25_run":      "data/google/runs/run.bm25_portuguese-msmarco.txt",
    "triples":       "data/triples.train.ids.small.tsv",                    # só IDs, 0,9 GB
}

def baixar(key):
    t0 = time.time()
    p = hf_hub_download(REPO, FILES[key], repo_type="dataset", local_dir="/content/mmarco")
    print(f"  {key:14s} ok ({os.path.getsize(p)/1e6:,.0f} MB, {time.time()-t0:.0f}s)")
    return p

print("Baixando arquivos (disco local do Colab, não o Drive)...")
paths = {k: baixar(k) for k in FILES if PREP_TRAIN or k not in ("triples", "train_queries")}

## 1. Perguntas dev, rótulos (qrels) e candidatas do BM25 oficial

In [ ]:
def ler_tsv_dict(path):
    d = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            k, v = line.rstrip("\n").split("\t", 1)
            d[k] = v
    return d

dev_queries = ler_tsv_dict(paths["dev_queries"])

qrels = defaultdict(set)                       # formato TREC: qid 0 pid 1
with open(paths["dev_qrels"]) as f:
    for line in f:
        qid, _, pid, rel = line.split()
        if int(rel) > 0:
            qrels[qid].add(pid)

# 1ª passada no run: só descobrir quais qids existem (o arquivo tem ~7 milhões de linhas)
run_qids = set()
with open(paths["bm25_run"]) as f:
    for line in f:
        run_qids.add(line.split("\t", 1)[0])

elegiveis = sorted(q for q in dev_queries if q in qrels and q in run_qids)
print(f"dev small: {len(dev_queries)} perguntas | com qrels e no run: {len(elegiveis)}")

if N_DEV_QUERIES is None or N_DEV_QUERIES >= len(elegiveis):
    amostra = set(elegiveis)                                               # dev small completo
else:
    amostra = set(random.Random(SEED).sample(elegiveis, N_DEV_QUERIES))   # amostra aleatória
print(f"perguntas avaliadas: {len(amostra)}")

# 2ª passada: guardar o ranking completo (até 1000) só das perguntas amostradas
ranking = defaultdict(list)
with open(paths["bm25_run"]) as f:
    for line in f:
        qid, pid, rank = line.rstrip("\n").split("\t")
        if qid in amostra:
            ranking[qid].append((int(rank), pid))
for qid in ranking:
    ranking[qid].sort()
prof = [len(v) for v in ranking.values()]
print(f"profundidade do run por pergunta: min={min(prof)} max={max(prof)}")

In [ ]:
dev_rows = []
for qid in sorted(amostra):
    pids = [pid for _, pid in ranking[qid]]
    pos = {pid: r for r, pid in enumerate(pids, start=1)}
    rel_ranks = [pos.get(p, -1) for p in sorted(qrels[qid])]   # -1 = fora do top-1000 do BM25
    dev_rows.append({
        "qid": qid,
        "query": dev_queries[qid],
        "relevant_pids": sorted(qrels[qid]),
        "bm25_rel_ranks": rel_ranks,
        "candidates": pids[:DEPTH],
    })
dev_df = pd.DataFrame(dev_rows)

# Teto do reranqueamento = Recall@K do 1º estágio (fraqueza 7 da v1)
def recall_at(ranks, k):
    return np.mean([sum(1 for r in rr if 0 < r <= k) / len(rr) for rr in ranks])
def mrr10_bm25(ranks):
    return np.mean([max([1 / r for r in rr if 0 < r <= 10], default=0.0) for rr in ranks])

print("BM25 oficial na amostra:")
print(f"  MRR@10 = {mrr10_bm25(dev_df.bm25_rel_ranks):.4f}  (artigo mMARCO, dev completo, pt: 0.152)")
for k in (10, 25, 50, 100, 1000):
    print(f"  Recall@{k:<4d} = {recall_at(dev_df.bm25_rel_ranks, k):.4f}   <- teto de um reranqueador com K={k}")
print(f"perguntas com mais de 1 passagem relevante: {(dev_df.relevant_pids.str.len() > 1).sum()}")

## 2. Triplas de treino (amostra aleatória por *reservoir sampling*)

O arquivo tem dezenas de milhões de triplas `(qid, pid_positiva, pid_negativa)`. Lemos uma vez e mantemos uma
amostra aleatória uniforme de tamanho fixo, sem carregar tudo na memória. As negativas dessas triplas são
**negativos difíceis** (vieram do BM25), que é o que ensina um reranqueador (LEARN.md, Parte 5.5).

In [ ]:
if PREP_TRAIN:
    N_TOTAL = N_TRAIN_TRIPLES + N_VAL_TRIPLES
    rng = random.Random(SEED)
    reservoir, n_lidas = [], 0
    t0 = time.time()
    with open(paths["triples"]) as f:
        for line in f:
            n_lidas += 1
            if len(reservoir) < N_TOTAL:
                reservoir.append(line)
            else:
                j = rng.randrange(n_lidas)
                if j < N_TOTAL:
                    reservoir[j] = line
    print(f"{n_lidas:,} triplas lidas em {time.time()-t0:.0f}s; amostradas {len(reservoir):,}")

    triples = [tuple(l.rstrip("\n").split("\t")) for l in reservoir]
    rng.shuffle(triples)
    val_triples, train_triples = triples[:N_VAL_TRIPLES], triples[N_VAL_TRIPLES:]
    del reservoir
else:
    triples, train_triples, val_triples = [], [], []
    n_lidas = json.load(open(f"{DATA}/meta.json")).get("n_triples_in_file") if os.path.exists(f"{DATA}/meta.json") else None
    print("triplas de treino já preparadas: etapa pulada")


## 3. Textos: consultas de treino e passagens (uma passada pela coleção de 8,8 milhões)

In [ ]:
train_qids = {t[0] for t in triples}
train_queries = {}
if PREP_TRAIN:
    with open(paths["train_queries"], encoding="utf-8") as f:
        for line in f:
            qid, text = line.rstrip("\n").split("\t", 1)
            if qid in train_qids:
                train_queries[qid] = text

needed = set()
for t in triples:
    needed.update(t[1:])
for row in dev_rows:
    needed.update(row["candidates"]); needed.update(row["relevant_pids"])
print(f"passagens necessárias: {len(needed):,}")

passages, t0 = {}, time.time()
with open(paths["collection"], encoding="utf-8", errors="replace") as f:   # 'replace' em vez de 'ignore': não some com bytes em silêncio
    for line in f:
        pid, text = line.rstrip("\n").split("\t", 1)
        if pid in needed:
            passages[pid] = text
print(f"coleção lida em {time.time()-t0:.0f}s; encontradas {len(passages):,}")
faltando = needed - passages.keys()
assert not faltando, f"{len(faltando)} passagens não encontradas, ex.: {list(faltando)[:5]}"

In [ ]:
# Correção de codificação (fraqueza 12 da v1)
n_fix = {"passagens": 0, "consultas": 0}
if FIX_TEXT:
    for pid, txt in passages.items():
        novo = ftfy.fix_text(txt)
        if novo != txt:
            passages[pid] = novo; n_fix["passagens"] += 1
    for d in (train_queries,):
        for k, txt in d.items():
            novo = ftfy.fix_text(txt)
            if novo != txt:
                d[k] = novo; n_fix["consultas"] += 1
    fixed_q = dev_df["query"].map(ftfy.fix_text)
    n_fix["consultas"] += int((fixed_q != dev_df["query"]).sum())
    dev_df["query"] = fixed_q
print("textos alterados pelo ftfy:", n_fix)

In [ ]:
def para_df(trs, split):
    return pd.DataFrame({
        "qid": [t[0] for t in trs],
        "query": [train_queries[t[0]] for t in trs],
        "pos_text": [passages[t[1]] for t in trs],
        "neg_text": [passages[t[2]] for t in trs],
        "split": split,
    })
if PREP_TRAIN:
    train_df = pd.concat([para_df(train_triples, "train"), para_df(val_triples, "val")], ignore_index=True)
    train_df.to_parquet(f"{DATA}/train_triples.parquet", index=False)
else:
    print("train_triples.parquet já existe: mantido")

dev_pids = set()
for row in dev_rows:
    dev_pids.update(row["candidates"]); dev_pids.update(row["relevant_pids"])
pass_df = pd.DataFrame({"pid": sorted(dev_pids)}).assign(text=lambda d: d.pid.map(passages))

dev_df.to_parquet(f"{DATA}/dev_queries.parquet", index=False)
pass_df.to_parquet(f"{DATA}/dev_passages.parquet", index=False)

import torch, transformers
meta = {
    "created": time.strftime("%Y-%m-%d %H:%M:%S"), "seed": SEED,
    "config": dict(N_DEV_QUERIES=N_DEV_QUERIES, DEPTH=DEPTH, N_TRAIN_TRIPLES=N_TRAIN_TRIPLES,
                   N_VAL_TRIPLES=N_VAL_TRIPLES, FIX_TEXT=FIX_TEXT),
    "files": FILES, "n_triples_in_file": n_lidas, "ftfy_changed": n_fix,
    "bm25_mrr10_sample": float(mrr10_bm25(dev_df.bm25_rel_ranks)),
    "bm25_recall": {k: float(recall_at(dev_df.bm25_rel_ranks, k)) for k in (10, 25, 50, 100, 1000)},
    "versions": {"python": platform.python_version(), "pandas": pd.__version__, "ftfy": ftfy.__version__,
                 "torch": torch.__version__, "transformers": transformers.__version__},
}
json.dump(meta, open(f"{DATA}/meta.json", "w"), indent=2, ensure_ascii=False)
print(json.dumps(meta, indent=2, ensure_ascii=False))
print("\nPronto. Arquivos em", DATA)